# Mandarin Inference, Evaluation & Rollback Pipeline — Sprint 51, Task B

Independent pipeline that loads a candidate checkpoint, compares it against a
DSPy/GEPA baseline (reconstructed, clearly labeled and versioned), scores
both with a Qwen3 judge calibrated on the Sprint 50 golden judge-calibration
set, runs 8 automated readiness checks, verifies rollback, and produces
`advance` / `review_revise` / `reject_keep_baseline`.

This pipeline is fully runnable **offline / CPU-only** by default (judge
mock mode + a candidate fixture), so you can validate everything here
without a GPU. A later section shows how to switch to the real Qwen3-8B/4B
judge and a real candidate checkpoint when you have GPU time.


## 1. Environment setup

In [ ]:
!nvidia-smi

In [ ]:
# --- Option A: clone your GitHub repo (recommended for the "exact GitHub link" deliverable) ---
# !git clone https://github.com/<your-org>/<your-repo>.git
# %cd <your-repo>

# --- Option B: upload the provided mandarin_eval_pipeline.zip directly to this Colab session ---
import os, zipfile

ZIP_NAME = "mandarin_eval_pipeline.zip"

if not os.path.exists(ZIP_NAME) and not os.path.exists("mandarin_eval_pipeline"):
    print(f"'{ZIP_NAME}' not found in {os.getcwd()} — opening the upload dialog.")
    print("Select mandarin_eval_pipeline.zip from your computer.")
    from google.colab import files
    uploaded = files.upload()
    zips = [f for f in uploaded if f.endswith(".zip")]
    if zips:
        ZIP_NAME = zips[0]

if os.path.exists("mandarin_eval_pipeline"):
    print("mandarin_eval_pipeline/ already present, skipping extraction.")
elif os.path.exists(ZIP_NAME):
    with zipfile.ZipFile(ZIP_NAME) as z:
        z.extractall(".")
    print(f"Extracted {ZIP_NAME}.")
else:
    raise FileNotFoundError(f"Still no '{ZIP_NAME}' in {os.getcwd()} after the upload prompt.")

assert os.path.exists("mandarin_eval_pipeline"), "Extraction did not produce a mandarin_eval_pipeline/ folder."
%cd mandarin_eval_pipeline
!ls

## 2. Install dependencies

Uses `requirements-cuda.txt` for a real (non-mock) Qwen3 judge run on GPU.
For a mock-mode/offline run, plain `requirements.txt` is enough — mock mode
needs no GPU libraries at all, only `pyyaml`.

Note: if you later switch to a real Qwen3 judge and hit
`ImportError: Found an incompatible version of torchao` from `peft`'s LoRA
dispatch (a real issue encountered building the companion Task A pipeline
on Colab), the fix is the same: `pip uninstall -y torchao` — included below
pre-emptively.


In [ ]:
!pip install -q -r requirements-cuda.txt
!pip uninstall -y torchao -q

## 3. Run the pipeline — mock judge, fixture candidate (fully offline, default)

This is the safest first run: no GPU, no model downloads, no network calls.
The judge is a deterministic offline scorer (clearly labeled `mock_mode` in
every output file, never confused with a real Qwen3 judgment), and the
candidate is a versioned fixture standing in for a real checkpoint.

This fixture intentionally has PLANTED DEFECTS (a memorization leak and a
synthetic safety-sentinel violation) so you can see the pipeline correctly
catch them and return `reject_keep_baseline`.


In [ ]:
import subprocess, sys

def run_pipeline(overrides, run_name):
    cmd = [sys.executable, "run_pipeline.py", "--config", "config/default_config.yaml",
           "--set", f"run.run_name={run_name}"]
    for o in overrides:
        cmd += ["--set", o]
    print(" ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print("----- STDERR -----")
        print(result.stderr)
    print("Return code:", result.returncode)
    return result

result = run_pipeline([], "reject_demo")

## 4. Run the two other decision outcomes

The same pipeline, pointed at two other versioned candidate fixtures, to
show all three possible decisions:
- `clean_demo` → candidate has no defects at all → **advance**
- `review_demo` → candidate has only non-critical defects (format/wrong-
  language/failure-rate) → **review_revise**


In [ ]:
run_pipeline([
    "candidate.fixture.path=data/fixtures/candidate_fixture_clean.jsonl",
    "candidate.fixture.fixture_version=candidate-fixture-clean-v1",
    "checks.safety.fixture_path=data/fixtures/safety_probes_clean.jsonl",
    "checks.domain_regression.fixture_path=data/fixtures/domain_regression_probes_clean.jsonl",
], "clean_demo")

In [ ]:
run_pipeline([
    "candidate.fixture.path=data/fixtures/candidate_fixture_review.jsonl",
    "candidate.fixture.fixture_version=candidate-fixture-review-v1",
    "checks.safety.fixture_path=data/fixtures/safety_probes_clean.jsonl",
    "checks.domain_regression.fixture_path=data/fixtures/domain_regression_probes_clean.jsonl",
], "review_demo")

In [ ]:
import json
for run in ["reject_demo", "clean_demo", "review_demo"]:
    r = json.load(open(f"outputs/{run}/readiness_report.json"))
    print(f"{run:15s} -> {r['decision']:22s} | critical_flags={r['critical_flags']}")

## 5. (Optional) Real Qwen3 judge + real candidate checkpoint

Requires a GPU (T4 or better) and network access to the Hugging Face Hub.
Qwen3-8B is primary; Qwen3-4B is the local fallback if 8B fails to load
(OOM, unavailable). The judge is calibrated on the Sprint 50 golden set
BEFORE it is allowed to score anything — if calibration doesn't clear the
agreement threshold, the pipeline refuses to freeze `judge_registry.json`
and stops, by design.

To also use the real Task A candidate checkpoint instead of the fixture,
set `candidate.use_fixture=false` (its `source_dir` already points at
`data/reference_candidate/`, the real Task A submission bundled in this zip).


In [ ]:
# Uncomment to run with the real judge (needs GPU + network):
# run_pipeline([
#     "judge.mock_mode=false",
#     "candidate.use_fixture=false",
# ], "real_judge_demo")

## 6. Inspect the frozen judge registry

In [ ]:
print(json.dumps(json.load(open("policies/judge_registry.json")), indent=2, ensure_ascii=False))

## 7. Inspect the frozen readiness policy

In [ ]:
print(json.dumps(json.load(open("policies/readiness_policy.json")), indent=2, ensure_ascii=False))

## 8. Read the human-readable readiness report

In [ ]:
print(open("outputs/reject_demo/readiness_report.md", encoding="utf-8").read())

## 9. Inspect rollback evidence

In [ ]:
print(json.dumps(json.load(open("outputs/reject_demo/rollback_evidence.json")), indent=2, ensure_ascii=False))

## 10. Package outputs for submission

Zips configs, policies, reports, and evidence together for download.


In [ ]:
import shutil
shutil.copytree("policies", "outputs/reject_demo/policies_snapshot", dirs_exist_ok=True)
zip_path = shutil.make_archive("readiness_pipeline_submission", "zip", "outputs")
print("Packaged:", zip_path)

from google.colab import files
files.download(zip_path)